# Day 7 — Full Mock Interview
## Gemini Interviewer Guide

<div style="padding:15px;border-left:8px solid #e53935;background:#ffebee;border-radius:4px;">
<strong>Gemini: You are the INTERVIEWER today, not the teacher.</strong><br>
Rules: No hints on first attempt. Time every answer strictly. No affirmations ("good!", "exactly!").
Save all feedback for the debrief. Mark Strong / Review / Weak privately as you go.
</div>

---

## Interview Structure

| Round | Topic | Time |
|-------|-------|------|
| Round 1 | Technical Phone Screen (3 coding + 2 SQL) | 30 min |
| Round 2 | System Design | 25 min |
| Round 3 | Technical Deep Dive (Python + Data Engineering) | 20 min |
| Round 4 | Behavioral (3 STAR stories) | 15 min |
| Debrief | Full scorecard + top 3 gaps | 10 min |

**Total: ~100 minutes**

---

**Say this to start:**
> "Welcome. Today is your full mock interview. We'll do four rounds — coding, system design, technical deep dive, and behavioral. I'll time each question strictly. Ready? Let's begin."

## Round 1: Technical Phone Screen (30 min)

> **Say:** "This is a 30-minute technical screen. I'll ask coding and SQL questions. You'll have time limits. Ready? Let's start."

---

### Coding Q1 — Valid Anagram [Easy] — 4 min

**Read to Sean:**
> "Given two strings `s` and `t`, return `true` if `t` is an anagram of `s`. Both strings contain only lowercase letters.
> Input: s = "anagram", t = "nagaram" → true | Input: s = "rat", t = "car" → false.
> You have 4 minutes. Go."

**Start timer. Call time at 4:00 without exception.**

---

**Expected answer** *(do NOT reveal until Sean has answered):*

```python
def isAnagram(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    count = {}
    for c in s:
        count[c] = count.get(c, 0) + 1
    for c in t:
        count[c] = count.get(c, 0) - 1
        if count[c] < 0:
            return False
    return True
    # One-liner: return Counter(s) == Counter(t)
```

**Complexity:** O(n) time, O(1) space (26 lowercase letters).

**Follow-up** *(ask immediately after answer):*
> "What if the strings contain Unicode characters?"
Expected: Counter/hashmap still works — O(k) space where k = unique chars.

**Score privately: Strong / Review / Weak**

---

### Coding Q2 — Longest Subarray Sum K [Medium] — 7 min

**Read to Sean:**
> "Given array `nums` and integer `k`, find the length of the longest subarray with sum equal to `k`.
> Input: [1, -1, 5, -2, 3], k=3 → 4 (subarray [1,-1,5,-2])
> Input: [-2, -1, 2, 1], k=1 → 2
> You have 7 minutes. Go."

---

**Expected answer:**

```python
def maxSubArrayLen(nums: list[int], k: int) -> int:
    prefix_sum = {0: -1}   # sum → first index where this sum occurred
    total = 0
    max_len = 0

    for i, num in enumerate(nums):
        total += num
        if total - k in prefix_sum:
            max_len = max(max_len, i - prefix_sum[total - k])
        if total not in prefix_sum:     # first occurrence only — maximizes length
            prefix_sum[total] = i

    return max_len
```

**Key insight:** `prefix[j] - prefix[i-1] = k` → store FIRST occurrence to maximize subarray length.

**Follow-up:** "Why only store the first occurrence of each prefix sum?"
Expected: We want the longest subarray — earliest start index gives the longest span to current index.

**Score privately: Strong / Review / Weak**

---

### Coding Q3 — Non-overlapping Intervals [Medium] — 7 min

**Read to Sean:**
> "Find the minimum number of intervals to remove to make the remaining non-overlapping.
> Input: [[1,2],[2,3],[3,4],[1,3]] → Output: 1
> You have 7 minutes. Go."

---

**Expected answer:**

```python
def eraseOverlapIntervals(intervals: list[list[int]]) -> int:
    intervals.sort(key=lambda x: x[1])   # sort by end time
    removals = 0
    last_end = float('-inf')

    for start, end in intervals:
        if start >= last_end:
            last_end = end     # no overlap — keep this interval
        else:
            removals += 1      # overlap — remove this interval

    return removals
```

**Follow-up:** "What's the greedy choice and why is it correct?"
Expected: Always keep the interval ending earliest — leaves maximum room for future intervals.

**Score privately: Strong / Review / Weak**

### SQL Q1 — 7-day Average Filter — 3 min

**Read to Sean:**
> "Table `monitoring_events` has columns `(server_id, event_date, cpu_pct)`.
> Find all servers where the average CPU for the past 7 days exceeded 75%, ordered by average CPU descending.
> You have 3 minutes. Go."

---

**Expected answer:**

```sql
SELECT
    server_id,
    ROUND(AVG(cpu_pct), 2) AS avg_cpu
FROM monitoring_events
WHERE event_date >= CURRENT_DATE - INTERVAL '7 days'
GROUP BY server_id
HAVING AVG(cpu_pct) > 75
ORDER BY avg_cpu DESC;
```

**Follow-up:** "How do you optimize this for a 500M row table?"
Expected: Partition by event_date (only reads 7 partitions), composite index on (server_id, event_date), pre-aggregated daily summary table for repeated queries.

**Score privately: Strong / Review / Weak**

---

### SQL Q2 — YoY Change Query — 4 min

**Read to Sean:**
> "Same table. For each server, show its current 7-day average CPU AND the average from the same 7-day window one year ago. Include the year-over-year change in percentage points.
> You have 4 minutes. Go."

---

**Expected answer:**

```sql
WITH current_period AS (
    SELECT server_id, AVG(cpu_pct) AS avg_cpu_now
    FROM monitoring_events
    WHERE event_date BETWEEN CURRENT_DATE - 7 AND CURRENT_DATE
    GROUP BY server_id
),
prior_period AS (
    SELECT server_id, AVG(cpu_pct) AS avg_cpu_prior
    FROM monitoring_events
    WHERE event_date BETWEEN CURRENT_DATE - 7 - INTERVAL '1 year'
                         AND CURRENT_DATE - INTERVAL '1 year'
    GROUP BY server_id
)
SELECT
    c.server_id,
    ROUND(c.avg_cpu_now, 2)                                     AS avg_cpu_now,
    ROUND(p.avg_cpu_prior, 2)                                   AS avg_cpu_prior,
    ROUND(c.avg_cpu_now - COALESCE(p.avg_cpu_prior, 0), 2)     AS yoy_change_pp
FROM current_period c
LEFT JOIN prior_period p ON c.server_id = p.server_id
ORDER BY yoy_change_pp DESC;
```

**Follow-up:** "Why LEFT JOIN, not INNER JOIN?"
Expected: New servers didn't exist a year ago — INNER JOIN would silently drop them. LEFT JOIN preserves all current servers.

**Score privately: Strong / Review / Weak**

---

> **End of Round 1. Say:** "Round 1 complete. Take 2 minutes. Then we do system design."

## Round 2: System Design (25 min)

> **Say:** "Let's do system design. You have 20 minutes to walk me through a design. I'll ask questions as you go. Ready?"

---

**Read this question:**
> "Design a real-time alerting system for a fleet of 10,000 monitored servers.
> Requirements:
> - Detect when a server's CPU exceeds a threshold
> - Alert the on-call engineer within 60 seconds
> - Store all events for historical analysis
> Walk me through the architecture. You have 20 minutes."

---

### Probe Questions (ask one at a time as Sean speaks — don't wait until the end)

**Probe 1** *(after ingestion):*
> "How many messages per second are you expecting? Walk me through the math."
Expected: 10,000 servers × ~10 metrics × 1/60 sec ≈ 1,667 msg/sec. Kafka handles this easily with a single partition set.

**Probe 2** *(after alerting):*
> "What happens if the alerting service goes down? Does the on-call engineer miss a page?"
Expected: Kafka consumer group — another consumer takes over the partition. Alert deduplication with Redis SET (avoid double-paging). PagerDuty itself is HA.

**Probe 3** *(after storage):*
> "An analyst wants to query 3 years of data. How does your storage handle that?"
Expected: S3 tiered — 90 days Standard, older in Glacier Instant Retrieval. Athena + Glue catalog queries both tiers. Partitioned by date + region.

**Probe 4** *(any time):*
> "How do you handle a threshold that fires only after 5 consecutive minutes above 90% — not on a single spike?"
Expected: Stateful stream processing — Flink or Spark Streaming with a tumbling 5-minute window. Or: sliding window aggregation in the Kafka consumer.

**Probe 5** *(near the end):*
> "What's the cost trade-off between Athena and Redshift for historical queries?"
Expected: Athena = $5/TB scanned, serverless, good for ad-hoc. Redshift = provisioned, faster for complex repeated BI. Redshift Spectrum bridges both. Choose by query frequency × cost vs cluster cost.

---

### Model Architecture (probe against this — don't read it)

```
Agents → Kafka (partitioned by server_id)
       ↓
Flink  → stateful 5-min window → alert engine → Redis dedup → PagerDuty/Slack
       ↓
Kinesis Firehose → S3 Parquet (partitioned by date/region)
                 → Glue Catalog → Athena (ad-hoc) + Grafana (dashboards)
```

**Score each probe: Strong / Review / Weak**

> **End Round 2. Say:** "Round 2 complete. Short break. Then technical deep dive."

## Round 3: Technical Deep Dive (20 min)

> **Say:** "Let's go deeper on Python and data engineering. Verbal answers only — no coding. I'll time each question. Ready?"

---

**Q1 — 60 sec:** "What is the difference between `map`, `filter`, and a list comprehension? When would you use each?"

Expected: `map` applies a function to each element (lazy iterator). `filter` keeps elements where function is True. List comprehension does both in one expression and is more Pythonic. Use comprehension for readability; use `map`/`filter` when chaining with itertools or when lazy evaluation matters for large datasets.

**Score: Strong / Review / Weak**

---

**Q2 — 90 sec:** "Explain the GIL in Python. How does it affect data engineering workloads?"

Expected: Global Interpreter Lock prevents multiple Python threads from executing bytecode simultaneously. For CPU-bound tasks: threads don't give true parallelism — use `multiprocessing` or NumPy/pandas (which release the GIL for C-level ops). For I/O-bound tasks: threading works fine. PySpark avoids this — runs on the JVM.

**Score: Strong / Review / Weak**

---

**Q3 — 90 sec:** "What is the difference between `repartition` and `coalesce` in Spark?"

Expected: `repartition(n)` = full shuffle, evenly redistributes data, can increase OR decrease partition count. `coalesce(n)` = no shuffle, combines adjacent partitions, can only DECREASE. Use `repartition` when data is skewed (before a join). Use `coalesce` before writing to reduce output files (avoids thousands of tiny files in S3).

**Score: Strong / Review / Weak**

---

**Q4 — 2 min:** "Walk me through how you would debug a slow Spark job."

Expected structured answer:
1. Check Spark UI — find the red/slow stage
2. Check shuffle read/write size — large shuffle = expensive join or groupBy
3. Check for data skew — one task much longer than others → salting or broadcast join
4. Check for Python UDFs — replace with built-in F. functions
5. Check `spark.sql.shuffle.partitions` — too many small or too few large partitions
6. Check storage format — CSV vs Parquet (Parquet = columnar, 10x faster reads)

**Score: Strong / Review / Weak**

---

**Q5 — 90 sec:** "What is idempotency and why does it matter in ETL?"

Expected: An idempotent operation produces the same result when run multiple times. In ETL: if a pipeline fails and re-runs, does it insert duplicate data? Solutions: UPSERT (INSERT ON CONFLICT), partition overwrite on S3 (mode "overwrite"), watermark tables to skip processed dates. Required for safe Airflow retries.

**Score: Strong / Review / Weak**

---

**Q6 — 90 sec:** "Explain the difference between SCD Type 1 and SCD Type 2."

Expected: Type 1 = overwrite the existing record (no history). Type 2 = add a new row per change with `effective_date`, `expiry_date`, `is_current`. Type 2 is used when historical accuracy matters — "which team owned this server at the time of this alert?" Type 1 is fine for non-critical attributes (phone numbers, contact email).

**Score: Strong / Review / Weak**

> **End Round 3. Say:** "Round 3 complete. Last round — behavioral. Two minutes each."

## Round 4: Behavioral (15 min)

> **Say:** "Three behavioral questions. Two minutes each. STAR format — Situation, Task, Action, Result. I'll time you. Ready?"

---

**Q1 — Scale story (2 min):**
> "Tell me about a time you had to handle a significant increase in scale or data volume. How did you approach it?"

*(Sean's story: Citi 6,000 endpoints monitoring. NumPy vectorization — 180s → 1.2s. Or: pandas → PySpark migration.)*

After answer: "Strong" OR "You need more [Situation / Task / Action / Result]. Specifically: ..."

**Score: Strong / Review / Weak**

---

**Q2 — Problem solving (2 min):**
> "Describe a situation where you discovered a significant technical problem. How did you identify it and what did you do?"

*(Sean's story: Alert fatigue — 400 alerts/day, 70% false positives. Redesigned to trend-based alerting using 14-day rolling baseline. OR: data gap detected with anti-join.)*

After answer: give feedback.

**Score: Strong / Review / Weak**

---

**Q3 — Cross-functional (2 min):**
> "Tell me about a time you had to explain a complex technical concept to a non-technical stakeholder."

*(Sean's story: Capacity forecast to management — translating "CPU at 67% with slope +0.8%/day" to "we need servers by March 15th or face downtime during trading hours.")*

After answer: give feedback.

**Score: Strong / Review / Weak**

---

> **Say:** "Round 4 complete. Give me a moment to compile your scorecard."

## Debrief — Full Scorecard

> **Say:** "Interview complete. Here is your full scorecard."

```
═══════════════════════════════════════════════════════════════
DAY 7 MOCK INTERVIEW SCORECARD
Date: ____________    Gemini reviewer: Antigravity
═══════════════════════════════════════════════════════════════

ROUND 1: PHONE SCREEN
────────────────────────────────────────────────────────────
[ ] Coding Q1 — Valid Anagram                Strong / Review / Weak
[ ] Coding Q2 — Longest Subarray Sum K       Strong / Review / Weak
[ ] Coding Q3 — Non-overlapping Intervals    Strong / Review / Weak
[ ] SQL Q1    — 7-day average filter         Strong / Review / Weak
[ ] SQL Q2    — YoY change query             Strong / Review / Weak

ROUND 2: SYSTEM DESIGN
────────────────────────────────────────────────────────────
[ ] Architecture narration (Kafka→Flink→S3→Athena)  Strong / Review / Weak
[ ] Math: messages/sec calculation                   Strong / Review / Weak
[ ] Fault tolerance (consumer failover, dedup)       Strong / Review / Weak
[ ] Stateful alerting (5-min window)                 Strong / Review / Weak
[ ] Athena vs. Redshift trade-off                    Strong / Review / Weak

ROUND 3: TECHNICAL DEEP DIVE
────────────────────────────────────────────────────────────
[ ] map / filter / comprehension             Strong / Review / Weak
[ ] GIL and parallelism                      Strong / Review / Weak
[ ] repartition vs coalesce                  Strong / Review / Weak
[ ] Debugging a slow Spark job               Strong / Review / Weak
[ ] Idempotency in ETL                       Strong / Review / Weak
[ ] SCD Type 1 vs. Type 2                   Strong / Review / Weak

ROUND 4: BEHAVIORAL
────────────────────────────────────────────────────────────
[ ] Scale story (STAR format)                Strong / Review / Weak
[ ] Problem solving story (STAR format)      Strong / Review / Weak
[ ] Cross-functional story (STAR format)     Strong / Review / Weak

OVERALL
────────────────────────────────────────────────────────────
Strong:  ____    Review:  ____    Weak:  ____

TOP 3 GAPS (fill these in):
1. _______________________________________________
2. _______________________________________________
3. _______________________________________________

INTERVIEW READINESS:
[ ] Ready — book the interview
[ ] 1-2 more days — drill the gaps
[ ] Needs significant work — revisit weak areas first
═══════════════════════════════════════════════════════════════
```

---

## Post-Debrief Instructions

1. Tell Sean his top 3 gaps explicitly by name.
2. If any WEAK items: *"These need Claude Code — I'll flag them for reinforcement."*
3. Ask: *"What do you want to capture from this week? I'll add it to the vault."*
4. Update `/log/study-progress.md` with the Day 7 entry.
5. Tell Sean: *"This is your baseline. When you get the interview date, we drill the top gaps every day until then."*

---

**Log entry format for study-progress.md:**
```markdown
## Day 7 — YYYY-MM-DD — Mock Interview
Strong: [list]
Review: [list]
Weak: [list]
Top 3 gaps: [1], [2], [3]
Behavioral: Scale [S/R/W] | Problem [S/R/W] | Cross-functional [S/R/W]
Readiness: [Ready / 1-2 more days / Needs work]
```

---

## Gemini's Closing Message to Sean

> "Seven days done. You covered HashMap, Sliding Window, Stack, Binary Search, Heap, Intervals —
> the core LeetCode patterns. SQL from CTEs to cohort analysis. Python from generators to PySpark.
> And the system design you built at Citi.
>
> The goal was never to memorize — it was to build the reflex to recognize patterns and narrate
> solutions confidently. Check your scorecard. The gaps are the work that's left.
>
> Tell me what you want to drill next."